In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("swynex_healthcare_hospital_raw.csv")

df = df.replace(["", " ", "N/A", "Unknown", "None", "null"], np.nan)
df = df.drop_duplicates().drop_duplicates(subset=["patient_id"], keep="first")

df["patient_name"] = df["patient_name"].astype(str).str.strip().str.title()
df["gender"] = df["gender"].astype(str).str.strip().str.lower().replace({
    "m": "Male", "male": "Male", "f": "Female", "female": "Female", "other": "Other"
})
df["medical_condition"] = df["medical_condition"].astype(str).str.strip().str.lower().replace({
    "diabetes": "Diabetes", "type-2 diabetes": "Diabetes",
    "hypertension": "Hypertension", "high bp": "Hypertension",
    "asthma": "Asthma", "cardiac": "Cardiac Care", "cardiac care": "Cardiac Care",
    "heart issue": "Cardiac Care", "pneumonia": "Pneumonia", "arthritis": "Arthritis"
})

df["billing_amount"] = pd.to_numeric(
    df["billing_amount"].astype(str).str.replace(r"[₹,]|INR", "", regex=True).str.strip(),
    errors="coerce"
).abs()
df["billing_amount"] = df["billing_amount"].fillna(df["billing_amount"].median())

for col in ["age", "satisfaction_score"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median()).astype(int)

df["insurance_provider"] = df["insurance_provider"].fillna("Self-Pay")
df["discharge_outcome"] = df["discharge_outcome"].fillna("Follow-Up Required")

adm = pd.to_datetime(df["admission_date"], format="mixed", dayfirst=True, errors="coerce")
dis = pd.to_datetime(df["discharge_date"], format="mixed", dayfirst=True, errors="coerce")
swap = dis < adm
df["admission_date"] = adm.where(~swap, dis).dt.strftime("%Y-%m-%d")
df["discharge_date"] = dis.where(~swap, adm).dt.strftime("%Y-%m-%d")
df["length_of_stay_days"] = (pd.to_datetime(df["discharge_date"]) - pd.to_datetime(df["admission_date"])).dt.days

df.to_csv("swynex_healthcare_hospital_cleaned.csv", index=False)
print("Done! Final rows:", len(df))

Done! Final rows: 850
